In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import h5py
import illustris_python as il
from matplotlib.colors import LogNorm

In [ ]:
zs = [0.0,0.1,0.2,0.3,0.4,0.5,0.7,1.0,1.5,2.0,3.01,4.01,5.0,6.01,7.01,8.01,9.0,10.0]
snaps_ids = [99,91,84,78,72,67,59,50,40,33,25,21,17,13,11,8,6,4]

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
})

basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"

L = 35.0 # cMpc/h
res = 128

dl  = L / res
print (f"The voxel width is {dl} cMpc/h")

norm_dens = []
voids = []
filaments = []
nodes = []
walls = []
dens = []

with h5py.File('/Users/users/roana/roana/BSc_Thesis/nexus_outputs.h5', 'r') as f:
    for red in zs:
        norm_dens.append(np.transpose(f[f'z_{red}']['normalized_density'][:], (1,2,0)))
        dens.append(np.transpose(f[f'z_{red}']['density'][:], (1,2,0)))
        voids.append(np.transpose(f[f'z_{red}']['voids'][:], (1,2,0)))
        filaments.append(np.transpose(f[f'z_{red}']['filaments'][:], (1,2,0)))
        nodes.append(np.transpose(f[f'z_{red}']['nodes'][:], (1,2,0)))
        walls.append(np.transpose(f[f'z_{red}']['walls'][:], (1,2,0)))
        print(f"Suessfully Loaded Stuff for z={red}")

n = len(norm_dens)
print(f"{n} snapshots")

In [ ]:
def huhuhehe(snap_curr):
    # 1. Setup paths, snapshots, and parameters
    basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"
    snap_init = 0   # Initial snapshot (z=20.05)
    #snap_curr = 99  # Current snapshot (z=0.0)
    part_type = 1   # 1 is for Dark Matter
    fraction = 0.01 # Process only 10% of the particles

    print(f"Working on {snap_curr}")

    # 2. Load Data (Only loading the specific fields we need to save memory)
    fields_init = ['ParticleIDs', 'Velocities']
    data_init = il.snapshot.loadSubset(basePath, snap_init, part_type, fields=fields_init)
    id_init =  data_init['ParticleIDs']
    v_init = data_init['Velocities']

    fields_curr = ['ParticleIDs', 'Velocities', 'Coordinates']
    data_curr = il.snapshot.loadSubset(basePath, snap_curr, part_type, fields=fields_curr)
    id_curr = data_curr['ParticleIDs']
    v_curr = data_curr['Velocities']
    pos_curr = data_curr['Coordinates']/1000


    # 3. Sort both arrays so the same particle is at the exact same index
    sort_init = np.argsort(id_init)
    sort_curr = np.argsort(id_curr)

    v_init_sorted = v_init[sort_init]
    v_curr_sorted = v_curr[sort_curr]
    pos_curr_sorted = pos_curr[sort_curr]

    # Free up memory from the unsorted arrays
    del data_init, data_curr, id_init, id_curr, v_init, v_curr, pos_curr, sort_init, sort_curr


    # 4. Subsample the particles
    num_particles = len(v_curr_sorted)
    num_sample = int(num_particles * fraction)

    # Randomly select indices without replacement
    np.random.seed(42) # Set seed so your plots are reproducible
    sample_indices = np.random.choice(num_particles, num_sample, replace=False)

    v_init_sampled = v_init_sorted[sample_indices]
    v_curr_sampled = v_curr_sorted[sample_indices]
    pos_curr_sampled = pos_curr_sorted[sample_indices]


    # 5. Compute the Dot Product and Magnitudes
    # np.sum with axis=1 multiplies element-wise and sums the x, y, z components
    dot_product = np.sum(v_init_sampled * v_curr_sampled, axis=1)

    # Calculate magnitudes (adding a tiny number 1e-10 to prevent division by zero errors)
    norm_init = np.linalg.norm(v_init_sampled, axis=1) + 1e-10
    norm_curr = np.linalg.norm(v_curr_sampled, axis=1) + 1e-10

    # Calculate cosine
    cos_theta = dot_product / (norm_init * norm_curr)


    # 6. Define a slice
    # Taking a slice along the Z-axis between 0.0 and 10.0 cMpc/h (converted to ckpc/h)
    z_min = 0.0 
    z_max = 10.0 

    # Extract Z coordinates and create a boolean mask
    z_coords = pos_curr_sampled[:, 2]
    slice_mask = (z_coords >= z_min) & (z_coords <= z_max)

    # Apply the mask
    x_slice = pos_curr_sampled[:, 0][slice_mask]
    y_slice = pos_curr_sampled[:, 1][slice_mask]
    cos_slice = cos_theta[slice_mask]

    return x_slice, y_slice, cos_slice

In [ ]:
def magnitude(snap_curr):
    # 1. Setup paths, snapshots, and parameters
    basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"
    snap_init = 0   # Initial snapshot (z=20.05)
    #snap_curr = 99  # Current snapshot (z=0.0)
    part_type = 1   # 1 is for Dark Matter
    fraction = 0.01 # Process only 10% of the particles

    print(f"Working on {snap_curr}")

    # 2. Load Data (Only loading the specific fields we need to save memory)
    fields_init = ['ParticleIDs', 'Velocities']
    data_init = il.snapshot.loadSubset(basePath, snap_init, part_type, fields=fields_init)
    id_init =  data_init['ParticleIDs']
    v_init = data_init['Velocities']

    fields_curr = ['ParticleIDs', 'Velocities', 'Coordinates']
    data_curr = il.snapshot.loadSubset(basePath, snap_curr, part_type, fields=fields_curr)
    id_curr = data_curr['ParticleIDs']
    v_curr = data_curr['Velocities']
    pos_curr = data_curr['Coordinates']/1000


    # 3. Sort both arrays so the same particle is at the exact same index
    sort_init = np.argsort(id_init)
    sort_curr = np.argsort(id_curr)

    v_init_sorted = v_init[sort_init]
    v_curr_sorted = v_curr[sort_curr]
    pos_curr_sorted = pos_curr[sort_curr]

    # Free up memory from the unsorted arrays
    del data_init, data_curr, id_init, id_curr, v_init, v_curr, pos_curr, sort_init, sort_curr


    # 4. Subsample the particles
    num_particles = len(v_curr_sorted)
    num_sample = int(num_particles * fraction)

    # Randomly select indices without replacement
    np.random.seed(42) # Set seed so your plots are reproducible
    sample_indices = np.random.choice(num_particles, num_sample, replace=False)

    v_init_sampled = v_init_sorted[sample_indices]
    mag_0 = np.sqrt(v_init_sampled[:,0]**2 + v_init_sampled[:,1]**2 + v_init_sampled[:,2]**2) + 1e-10
    v_curr_sampled = v_curr_sorted[sample_indices] 
    pos_curr_sampled = pos_curr_sorted[sample_indices]
    mag = np.sqrt(v_curr_sampled[:,0]**2 + v_curr_sampled[:,1]**2 + v_curr_sampled[:,2]**2) + 1e-10


    # 5. Compute the Dot Product and Magnitudes
    # np.sum with axis=1 multiplies element-wise and sums the x, y, z components
    factor = mag/mag_0

    # 6. Define a slice
    # Taking a slice along the Z-axis between 0.0 and 10.0 cMpc/h (converted to ckpc/h)
    z_min = 0.0 
    z_max = 10.0 

    # Extract Z coordinates and create a boolean mask
    z_coords = pos_curr_sampled[:, 2]
    slice_mask = (z_coords >= z_min) & (z_coords <= z_max)

    # Apply the mask
    x_slice = pos_curr_sampled[:, 0][slice_mask]
    y_slice = pos_curr_sampled[:, 1][slice_mask]
    factor_slice = factor[slice_mask]

    return x_slice, y_slice, factor_slice

In [ ]:
red = [ 20.05, 14.99, 11.98, 10.98, 10.00, 9.39, 9.00, 8.45, 8.01, 7.60, 7.24, 7.01, 6.49, 6.01,
    5.85, 5.53, 5.23, 5.00, 4.66, 4.43, 4.18, 4.01, 3.71, 3.49,
    3.28, 3.01, 2.90, 2.73, 2.58, 2.44, 2.32, 2.21, 2.10, 2.00,
    1.90, 1.82, 1.74, 1.67, 1.60, 1.53, 1.50, 1.41, 1.36, 1.30,
    1.25, 1.21, 1.15, 1.11, 1.07, 1.04, 1.00, 0.95, 0.92, 0.89,
    0.85, 0.82, 0.79, 0.76, 0.73, 0.70, 0.68, 0.64, 0.62, 0.60,
    0.58, 0.55, 0.52, 0.50, 0.48, 0.46, 0.44, 0.42, 0.40, 0.38,
    0.36, 0.35, 0.33, 0.31, 0.30, 0.27, 0.26, 0.24, 0.23, 0.21,
    0.20, 0.18, 0.17, 0.15, 0.14, 0.13, 0.11, 0.10, 0.08, 0.07,
    0.06, 0.05, 0.03, 0.02, 0.01, 0.00
]

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

import matplotlib.animation as animation
import matplotlib.cm as cm
import matplotlib.colors as mcolors

data = []

for i in range(100):
    data.append(huhuhehe(i))


fig, ax = plt.subplots(figsize=(10, 8))

# 3. Set up the colorbar ONCE using a ScalarMappable
# This prevents a new colorbar from being drawn on every frame
norm = mcolors.Normalize(vmin=-1, vmax=1)
sm = cm.ScalarMappable(cmap='cividis', norm=norm)
sm.set_array([]) 
cbar = fig.colorbar(sm, ax=ax, label=r'$\cos \theta$')

def update(frame):

    ax.clear()

    sc = plt.scatter(
        data[frame][0], 
        data[frame][1], 
        c=data[frame][2], 
        cmap='cividis', 
        vmin=-1, 
        vmax=1, 
        s=1.5,               # Adjust this size if the dots are too sparse/dense
        edgecolors='none'    # Removes borders for a cleaner look
    )

    plt.xlabel('x [cMpc/h]')
    plt.ylabel('y [cMpc/h]')
    
    plt.title(f"z={red[frame]}")
    plt.gca().set_aspect('equal', adjustable='box')

anim = animation.FuncAnimation(fig, update, frames=100, interval=50)

anim.save('cos_evol_new.mp4', writer='ffmpeg', fps=5)

print("2D Animation successfully saved!")

In [ ]:
snap = 99

x_slice, y_slice, cos_slice = huhuhehe(snap)

plt.figure(figsize=(10, 8))
sc = plt.scatter(
    x_slice, 
    y_slice, 
    c=cos_slice, 
    cmap='viridis', 
    vmin=-1, 
    vmax=1, 
    s=1.5,               # Adjust this size if the dots are too sparse/dense
    edgecolors='none'    # Removes borders for a cleaner look
)

plt.colorbar(sc, label=r'$\cos \theta$')
plt.xlabel('x [cMpc/h]')
plt.ylabel('y [cMpc/h]')
plt.gca().set_aspect('equal', adjustable='box') # Keep the geometry square

# Save and show
plt.savefig('velocity_cosine_slice_python.png', dpi=300, bbox_inches='tight')
print("Done! Plot saved as velocity_cosine_slice_python.png")

In [ ]:
snap = 99

x_slice, y_slice, cos_slice = magnitude(snap)

plt.figure(figsize=(10, 8))
sc = plt.scatter(
    x_slice, 
    y_slice, 
    c=cos_slice, 
    cmap='viridis',  
    s=1.5,               # Adjust this size if the dots are too sparse/dense
    edgecolors='none'    # Removes borders for a cleaner look
)

plt.colorbar(sc, label=r'$v_f/v_i$')
plt.xlabel('x [cMpc/h]')
plt.ylabel('y [cMpc/h]')
plt.gca().set_aspect('equal', adjustable='box') # Keep the geometry square

# Save and show
plt.savefig('speed_slice_python.png', dpi=300, bbox_inches='tight')
print("Done! Plot saved as velocity_cosine_slice_python.png")

In [ ]:
snap = 50

x_slice, y_slice, cos_slice = magnitude(snap)

plt.figure(figsize=(10, 8))
sc = plt.scatter(
    x_slice, 
    y_slice, 
    c=cos_slice, 
    cmap='viridis',  
    s=1.5,               # Adjust this size if the dots are too sparse/dense
    edgecolors='none'    # Removes borders for a cleaner look
)

plt.colorbar(sc, label=r'$v_f/v_i$')
plt.xlabel('x [cMpc/h]')
plt.ylabel('y [cMpc/h]')
plt.gca().set_aspect('equal', adjustable='box') # Keep the geometry square

# Save and show
print("Done!")

In [ ]:
data_arr = np.array(data)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

ax.plot(np.flip(zs),np.mean(data_arr[:,2,:], axis = 1))
ax.invert_xaxis()

In [ ]:
plt.hist(data_arr[17,2,:], bins = "scott", density = True);

In [ ]:
ids_z0 = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['ParticleIDs'])
coords_z0 = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['Coordinates'])/1000

print("At redshift z=0...")

bool_filament = filaments[0].astype(bool)
bool_wall = walls[0].astype(bool)
bool_node = nodes[0].astype(bool)
bool_void = voids[0].astype(bool)

grid_x = np.floor(coords_z0[:, 0] / dl).astype(int) % res
grid_y = np.floor(coords_z0[:, 1] / dl).astype(int) % res
grid_z = np.floor(coords_z0[:, 2] / dl).astype(int) % res

in_node_mask = bool_node[grid_x, grid_y, grid_z]

node_particle_ids = ids_z0[in_node_mask]
print(f"Total DM particles: {len(ids_z0)}")
print(f"Particles in nodes: {len(node_particle_ids)}")

in_filament_mask = bool_filament[grid_x, grid_y, grid_z]

filament_particle_ids = ids_z0[in_filament_mask]
print(f"Particles in filaments: {len(filament_particle_ids)}")

in_wall_mask = bool_wall[grid_x, grid_y, grid_z]

wall_particle_ids = ids_z0[in_wall_mask]
print(f"Particles in walls: {len(wall_particle_ids)}")

in_void_mask = bool_void[grid_x, grid_y, grid_z]

void_particle_ids = ids_z0[in_void_mask]
print(f"Particles in voids: {len(void_particle_ids)}")

In [ ]:
ids_0 = il.snapshot.loadSubset(basePath, 0, 'dm', fields = ['ParticleIDs'])
vels_0 = il.snapshot.loadSubset(basePath, 0, 'dm', fields = ['Velocities'])

# Assuming these variables are defined upstream
id_list = [ids_0, node_particle_ids, filament_particle_ids, wall_particle_ids, void_particle_ids]

mean_alignments = [[], [], [], [], []]
alignments_err_up = [[], [], [], [], []]
alignments_err_down = [[], [], [], [], []]
everything = [[], [], [], [], []]

vels_0_target_hat_list = []
id_list_new = []
vel_list_new = []

snap_initial = 0
snap_final = 99
snaps = range(snap_initial, snap_final + 1)

np.random.seed(42)

print("Computing preliminaries....")

# Pre-sort snapshot 0 globally to find initial properties
sort_idx_0 = np.argsort(ids_0)
sorted_ids_0 = ids_0[sort_idx_0]

for k in range(5):
    target = id_list[k]

    # --- FIX 1: Safe Subset Selection ---
    # Ensure you don't try to select 9000 particles if an environment (like voids) has fewer
    sample_size = min(9000, len(target))
    subset_indices = np.random.choice(len(target), size=sample_size, replace=False)
    ids_0_target = target[subset_indices]

    # --- FIX 2: Safe Binary Search Validation ---
    match_indices_0 = np.searchsorted(sorted_ids_0, ids_0_target)
    
    # Clip indices to prevent out-of-bounds errors
    match_indices_0 = np.clip(match_indices_0, 0, len(sorted_ids_0) - 1)
    original_indices_0 = sort_idx_0[match_indices_0]
    
    # Strictly filter out entries where searchsorted did not find an EXACT match
    valid_mask = (ids_0[original_indices_0] == ids_0_target)
    
    # Clean the arrays to only include verified matching particles
    original_indices_0 = original_indices_0[valid_mask]
    ids_0_target = ids_0_target[valid_mask]

    # Fetch and normalize velocities
    vels_0_target = vels_0[original_indices_0]
    vels_0_target_norms = np.linalg.norm(vels_0_target, axis=1)
    
    # Prevent division by zero if a particle is completely stationary
    vels_0_target_norms[vels_0_target_norms == 0] = 1.0
    vels_0_target_hat = vels_0_target / vels_0_target_norms[:, np.newaxis]
    
    vels_0_target_hat_list.append(vels_0_target_hat)
    id_list_new.append(ids_0_target)
    vel_list_new.append(vels_0_target)

# --- Snapshot Tracking Loop ---
for i in snaps:
    if (i % 10 == 0):
        print(f"Working on snaps {i} to {i+9}....")

    ids = il.snapshot.loadSubset(basePath, i, 'dm', fields = ['ParticleIDs'])
    vels = il.snapshot.loadSubset(basePath, i, 'dm', fields = ['Velocities'])

    sort_idx = np.argsort(ids)
    sorted_ids = ids[sort_idx]

    for k in range(5):
        # Match current snapshot against your strictly tracked baseline subset
        match_indices = np.searchsorted(sorted_ids, id_list_new[k])
        match_indices = np.clip(match_indices, 0, len(sorted_ids) - 1)
        original_indices = sort_idx[match_indices]

        # --- FIX 3: Dynamic Snapshot Validation ---
        # Particles can occasionally leave the simulation boundary or sublink tracking. 
        # Check that the snapshot particle matches your tracking ID.
        valid_mask = (ids[original_indices] == id_list_new[k])
        
        matched_vels = vels[original_indices]
        matched_vels_norms = np.linalg.norm(matched_vels, axis=1)
        matched_vels_norms[matched_vels_norms == 0] = 1.0
        matched_v_hat = matched_vels / matched_vels_norms[:, np.newaxis]

        # Use the valid mask to calculate dot products only for confirmed matching particle pairs
        v0_hat = vels_0_target_hat_list[k][valid_mask]
        vt_hat = matched_v_hat[valid_mask]

        dot_products = np.sum(v0_hat * vt_hat, axis=1)

        # Handle edge case if no particles matched
        if len(dot_products) == 0:
            mean, higher, lower = np.nan, np.nan, np.nan
        else:
            mean = np.mean(dot_products)
            higher_pts = dot_products[dot_products >= mean]
            lower_pts = dot_products[dot_products <= mean]
            
            higher = np.mean(higher_pts) - mean if len(higher_pts) > 0 else 0
            lower  = mean - np.mean(lower_pts) if len(lower_pts) > 0 else 0

        everything[k].append(dot_products)
        mean_alignments[k].append(mean)
        alignments_err_up[k].append(higher)
        alignments_err_down[k].append(lower)

print("Done!")

In [ ]:
everything_arr = np.array(everything)

In [ ]:
plt.hist(everything_arr[1,-1])

In [ ]:
def huhuhehe_new(snap_curr, target_ids):
    # 1. Setup paths, snapshots, and parameters
    basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"
    snap_init = 0   # Initial snapshot (z=20.05)
    part_type = 1   # 1 is for Dark Matter
    
    # I set fraction to 1.0. Since you are already filtering down to just the nodes,
    # you might want to plot all of them. Change this back to 0.01 if the nodes 
    # are still too dense to plot smoothly!
    fraction = 1.0  

    print("Loading initial data and filtering...")

    # 2. Load Data and filter IMMEDIATELY to save memory
    fields_init = ['ParticleIDs', 'Velocities']
    data_init = il.snapshot.loadSubset(basePath, snap_init, part_type, fields=fields_init)
    
    # Create a boolean mask of which initial particles belong to your structure
    mask_init = np.isin(data_init['ParticleIDs'], target_ids)
    
    id_init = data_init['ParticleIDs'][mask_init]
    v_init = data_init['Velocities'][mask_init]
    del data_init  # Free memory immediately

    print("Loading current data and filtering...")
    
    fields_curr = ['ParticleIDs', 'Velocities', 'Coordinates']
    data_curr = il.snapshot.loadSubset(basePath, snap_curr, part_type, fields=fields_curr)
    
    # Create a boolean mask of which current particles belong to your structure
    mask_curr = np.isin(data_curr['ParticleIDs'], target_ids)
    
    id_curr = data_curr['ParticleIDs'][mask_curr]
    v_curr = data_curr['Velocities'][mask_curr]
    pos_curr = data_curr['Coordinates'][mask_curr] / 1000.0
    del data_curr  # Free memory immediately

    print("Sorting filtered arrays to align Particle IDs...")

    # 3. Sort both FILTERED arrays so the same particle is at the exact same index
    sort_init = np.argsort(id_init)
    sort_curr = np.argsort(id_curr)

    v_init_sorted = v_init[sort_init]
    v_curr_sorted = v_curr[sort_curr]
    pos_curr_sorted = pos_curr[sort_curr]

    # Free up memory from the unsorted arrays
    del id_init, id_curr, v_init, v_curr, pos_curr, sort_init, sort_curr

    # 4. Subsample the particles (Optional now that we filtered)
    num_particles = len(v_curr_sorted)
    
    if fraction < 1.0:
        print(f"Subsampling: selecting {fraction*100}% of the node particles...")
        num_sample = int(num_particles * fraction)
        np.random.seed(42)
        sample_indices = np.random.choice(num_particles, num_sample, replace=False)
        
        v_init_sampled = v_init_sorted[sample_indices]
        v_curr_sampled = v_curr_sorted[sample_indices]
        pos_curr_sampled = pos_curr_sorted[sample_indices]
    else:
        # If fraction is 1.0, just skip the random choice step
        v_init_sampled = v_init_sorted
        v_curr_sampled = v_curr_sorted
        pos_curr_sampled = pos_curr_sorted

    print("Computing cosine of the velocities...")

    # 5. Compute the Dot Product and Magnitudes
    dot_product = np.sum(v_init_sampled * v_curr_sampled, axis=1)

    norm_init = np.linalg.norm(v_init_sampled, axis=1) + 1e-10
    norm_curr = np.linalg.norm(v_curr_sampled, axis=1) + 1e-10

    cos_theta = dot_product / (norm_init * norm_curr)

    print("Extracting spatial slice...")

    # 6. Define a slice
    z_min = 0.0 
    z_max = 35.0 

    z_coords = pos_curr_sampled[:, 2]
    slice_mask = (z_coords >= z_min) & (z_coords <= z_max)

    x_slice = pos_curr_sampled[:, 0][slice_mask]
    y_slice = pos_curr_sampled[:, 1][slice_mask]
    cos_slice = cos_theta[slice_mask]

    return x_slice, y_slice, cos_slice

In [ ]:
data = huhuhehe_new(99, node_particle_ids)

In [ ]:
plt.figure(figsize=(10, 8))
sc = plt.scatter(
    data[0], 
    data[1], 
    c=data[2], 
    cmap='viridis', 
    vmin=-1, 
    vmax=1, 
    s=1.5,               # Adjust this size if the dots are too sparse/dense
    edgecolors='none'    # Removes borders for a cleaner look
)

plt.colorbar(sc, label=r'$\cos \theta$')
plt.xlabel('x [cMpc/h]')
plt.ylabel('y [cMpc/h]')
plt.gca().set_aspect('equal', adjustable='box'); # Keep the geometry square

In [ ]:
data = huhuhehe_new(99, filament_particle_ids[::10])

plt.figure(figsize=(10, 8))
sc = plt.scatter(
    data[0], 
    data[1], 
    c=data[2], 
    cmap='viridis', 
    vmin=-1, 
    vmax=1, 
    s=1.5,               # Adjust this size if the dots are too sparse/dense
    edgecolors='none'    # Removes borders for a cleaner look
)

plt.colorbar(sc, label=r'$\cos \theta$')
plt.xlabel('x [cMpc/h]')
plt.ylabel('y [cMpc/h]')
plt.gca().set_aspect('equal', adjustable='box'); # Keep the geometry square

In [ ]:
data = huhuhehe_new(99, wall_particle_ids[::10])

plt.figure(figsize=(10, 8))
sc = plt.scatter(
    data[0], 
    data[1], 
    c=data[2], 
    cmap='viridis', 
    vmin=-1, 
    vmax=1, 
    s=1.5,               # Adjust this size if the dots are too sparse/dense
    edgecolors='none'    # Removes borders for a cleaner look
)

plt.colorbar(sc, label=r'$\cos \theta$')
plt.xlabel('x [cMpc/h]')
plt.ylabel('y [cMpc/h]')
plt.gca().set_aspect('equal', adjustable='box'); # Keep the geometry square

In [ ]:
data = huhuhehe_new(99, void_particle_ids[::100])

plt.figure(figsize=(10, 8))
sc = plt.scatter(
    data[0], 
    data[1], 
    c=data[2], 
    cmap='viridis', 
    vmin=-1, 
    vmax=1, 
    s=1.5,               # Adjust this size if the dots are too sparse/dense
    edgecolors='none'    # Removes borders for a cleaner look
)

plt.colorbar(sc, label=r'$\cos \theta$')
plt.xlabel('x [cMpc/h]')
plt.ylabel('y [cMpc/h]')
plt.gca().set_aspect('equal', adjustable='box'); # Keep the geometry square

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

import matplotlib.animation as animation
import matplotlib.cm as cm
import matplotlib.colors as mcolors

data = []

for i in range(100):
    data.append(magnitude(i))


fig, ax = plt.subplots(figsize=(10, 8))

# 3. Set up the colorbar ONCE using a ScalarMappable
# This prevents a new colorbar from being drawn on every frame
norm = mcolors.Normalize(vmin=-1, vmax=1)
sm = cm.ScalarMappable(cmap='cividis', norm=norm)
sm.set_array([]) 
cbar = fig.colorbar(sm, ax=ax, label=r'$v_f/v_i$')
minmin = np.min(data)
maxmax = np.max(data)

def update(frame):

    ax.clear()

    sc = plt.scatter(
        data[frame][0], 
        data[frame][1], 
        c=data[frame][2], 
        cmap='cividis', 
        vmin=-1, 
        vmax=1, 
        s=1.5,               # Adjust this size if the dots are too sparse/dense
        edgecolors='none'    # Removes borders for a cleaner look
    )

    plt.xlabel('x [cMpc/h]')
    plt.ylabel('y [cMpc/h]')
    
    plt.title(f"z={red[frame]}")
    plt.gca().set_aspect('equal', adjustable='box')

anim = animation.FuncAnimation(fig, update, frames=100, interval=50)

anim.save('mag_evol_new.mp4', writer='ffmpeg', fps=15)

print("2D Animation successfully saved!")

In [ ]:

sm = cm.ScalarMappable(cmap='inferno')
sm.set_array([]) 
cbar = fig.colorbar(sm, ax=ax, label=r'$v_f/v_i$')
# minmin = np.min(data)
# maxmax = np.max(data)

def update(frame):

    ax.clear()

    sc = plt.scatter(
        data[frame][0], 
        data[frame][1], 
        c=data[frame][2], 
        cmap='inferno', 
        vmin=-1, 
        vmax=1, 
        s=1.5,               # Adjust this size if the dots are too sparse/dense
        edgecolors='none'    # Removes borders for a cleaner look
    )

    plt.xlabel('x [cMpc/h]')
    plt.ylabel('y [cMpc/h]')
    
    plt.title(f"z={red[frame]}")
    plt.gca().set_aspect('equal', adjustable='box')

anim = animation.FuncAnimation(fig, update, frames=100, interval=50)

anim.save('mag_evol_new.mp4', writer='ffmpeg', fps=15)

print("2D Animation successfully saved!")

In [ ]:
data_arr = np.array(data[0])

In [ ]:
np.shape(data[2])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# 1. Calculate the absolute min and max of your color data across ALL frames.
# This ensures that "yellow" means the exact same value in frame 0 as it does in frame 99.
all_v_ratios = np.concatenate([frame_data[2] for frame_data in data])

v_min = np.percentile(all_v_ratios, 1)
v_max = np.percentile(all_v_ratios, 99)

# 2. Create the colorbar ONCE before the animation starts.
# We create a dummy scatter plot just to initialize the colorbar properly.
sc_dummy = ax.scatter([], [], c=[], cmap='inferno', vmin=v_min, vmax=v_max)
cbar = fig.colorbar(sc_dummy, ax=ax, label=r'$v_f/v_i$')

def update(frame):
    ax.clear()

    # 3. Add vmin and vmax to your scatter plot inside the loop
    sc = ax.scatter(
        data[frame][0], 
        data[frame][1], 
        c=data[frame][2], 
        cmap='inferno',  
        s=1.5,
        edgecolors='none',
        vmin=v_min,   # Lock the bottom of the color scale
        vmax=v_max    # Lock the top of the color scale
    )

    # Re-apply axis labels and settings (since ax.clear() wipes them)
    ax.set_xlabel('x [cMpc/h]')
    ax.set_ylabel('y [cMpc/h]')
    ax.set_aspect('equal', adjustable='box')

anim = animation.FuncAnimation(fig, update, frames=100, interval=50)

anim.save('mag_evol_new.mp4', writer='ffmpeg', fps=15)

print("2D Animation successfully saved!")

In [ ]:
data = []

for i in range(100):
    data.append(magnitude(i))